# 第九课：NumPy 广播（Broadcast）

本课依据菜鸟教程的 [NumPy 广播](https://www.runoob.com/numpy/numpy-broadcast.html) 整理，并补充形状对齐、行列向量、兼容性判断和标准化等实用示例。

广播让不同形状的数组在满足规则时直接进行逐元素运算，而不必手动复制数据。

In [1]:
import numpy as np
print("NumPy 版本：", np.__version__)

NumPy 版本： 2.5.3


## 1. 相同形状的逐元素运算

形状相同时，两个数组的对应位置直接运算。注意，`*` 是逐元素乘法，不是矩阵乘法；矩阵乘法使用 `@`。

In [2]:
a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30, 40])
print("相加：", a + b)
print("相乘：", a * b)

相加： [11 22 33 44]
相乘： [ 10  40  90 160]


## 2. 标量广播

标量可以看成没有轴的数组。它会被概念上扩展到数组的每一个位置。NumPy 不需要真的创建许多份标量。

In [3]:
prices = np.array([10.0, 25.0, 40.0])
print("加 5：", prices + 5)
print("打八折：", prices * 0.8)
print("是否超过 20：", prices > 20)

加 5： [15. 30. 45.]
打八折： [ 8. 20. 32.]
是否超过 20： [False  True  True]


## 3. 二维数组与一维数组广播

形状 `(4, 3)` 与 `(3,)` 可以广播。一维数组会与二维数组的最后一个轴对齐，因此相当于给每一行加上同一个长度为 3 的向量。

In [4]:
matrix = np.array([[0, 0, 0], [10, 10, 10], [20, 20, 20], [30, 30, 30]])
row_adjustment = np.array([0, 1, 2])
print("matrix shape：", matrix.shape)
print("row_adjustment shape：", row_adjustment.shape)
print("结果：\n", matrix + row_adjustment)

matrix shape： (4, 3)
row_adjustment shape： (3,)
结果：
 [[ 0  1  2]
 [10 11 12]
 [20 21 22]
 [30 31 32]]


## 4. 广播规则：从右向左比较形状

比较两个数组的形状时，从最后一个维度开始向左对齐。每个对应维度满足以下任一条件即可：

1. 两个维度相等；
2. 其中一个维度是 `1`；
3. 某个数组没有这个维度，可视为在左侧补 `1`。

输出形状取每个维度的较大值。

例：`(4, 3)` 与 `(3,)` 对齐为 `(4, 3)` 与 `(1, 3)`，结果是 `(4, 3)`。

In [5]:
shape_pairs = [
    ((4, 3), (3,)),
    ((4, 1), (1, 3)),
    ((2, 1, 4), (3, 1)),
]
for left, right in shape_pairs:
    print(left, "与", right, "→", np.broadcast_shapes(left, right))

(4, 3) 与 (3,) → (4, 3)
(4, 1) 与 (1, 3) → (4, 3)
(2, 1, 4) 与 (3, 1) → (2, 3, 4)


## 5. 行向量与列向量

一维数组 `(3,)` 默认与最后一轴对齐，是行方向。要沿行方向给每一行加一个不同值，需要把数组变成列向量 `(n, 1)`。

In [6]:
base = np.arange(12).reshape(4, 3)
per_column = np.array([100, 200, 300])        # shape (3,)
per_row = np.array([10, 20, 30, 40])[:, None] # shape (4, 1)
print("原数组：\n", base)
print("每列加不同值：\n", base + per_column)
print("每行加不同值：\n", base + per_row)

原数组：
 [[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]
每列加不同值：
 [[100 201 302]
 [103 204 305]
 [106 207 308]
 [109 210 311]]
每行加不同值：
 [[10 11 12]
 [23 24 25]
 [36 37 38]
 [49 50 51]]


列向量 `(n, 1)` 与行向量 `(m,)` 运算会得到 `(n, m)`，这可以创建所有两两组合。

In [7]:
rows = np.array([10, 20, 30])[:, None]  # (3, 1)
cols = np.array([1, 2, 3, 4])         # (4,)
print("外部加法：\n", rows + cols)
print("外部乘法：\n", rows * cols)

外部加法：
 [[11 12 13 14]
 [21 22 23 24]
 [31 32 33 34]]
外部乘法：
 [[ 10  20  30  40]
 [ 20  40  60  80]
 [ 30  60  90 120]]


## 6. 不兼容的形状

例如 `(2, 3)` 与 `(2,)` 从右侧比较时，最后一维 `3` 与 `2` 既不相等，也没有一个是 1，因此无法广播。下面用 `try/except` 展示错误而不中断 Notebook。

In [8]:
left = np.ones((2, 3))
right = np.ones(2)
try:
    result = left + right
except ValueError as error:
    print("广播失败：", error)

广播失败： operands could not be broadcast together with shapes (2,3) (2,) 


如果这两个值本来是要分别作用于两行，应将 `(2,)` 改成 `(2, 1)`。关键是明确数据的语义，而不是为了消除错误随意加轴。

In [9]:
right_column = right[:, None]
print("修正后的 shape：", right_column.shape)
print(left + right_column)

修正后的 shape： (2, 1)
[[2. 2. 2.]
 [2. 2. 2.]]


## 7. 广播与 `tile()` 的区别

`np.tile()` 会明确构造重复数组，通常占用更多内存。广播只是让运算按扩展后的形状进行，通常更简洁、高效。只有确实需要保存重复数据时才使用 `tile()`。

In [10]:
small = np.array([1, 2, 3])
tiled = np.tile(small, (4, 1))
broadcast_view = np.broadcast_to(small, (4, 3))
print("tile：\n", tiled)
print("broadcast_to：\n", broadcast_view)
print("tile 拥有的数据字节数：", tiled.nbytes)
print("broadcast_to 结果是否可写：", broadcast_view.flags.writeable)

tile：
 [[1 2 3]
 [1 2 3]
 [1 2 3]
 [1 2 3]]
broadcast_to：
 [[1 2 3]
 [1 2 3]
 [1 2 3]
 [1 2 3]]
tile 拥有的数据字节数： 96
broadcast_to 结果是否可写： False


## 8. 实际应用：按列标准化

对形状 `(样本数, 特征数)` 的数据，`mean(axis=0)` 和 `std(axis=0)` 得到每列统计量，形状为 `(特征数,)`。它们可以广播到每一行。

In [12]:
data = np.array([[10.0, 100.0], [20.0, 120.0], [30.0, 140.0]])
print("原始数据：\n", data)
column_mean = data.mean(axis=0)
column_std = data.std(axis=0)
standardized = (data - column_mean) / column_std
print("均值：", column_mean, "shape=", column_mean.shape)
print("标准差：", column_std)
print("标准化结果：\n", standardized)
print("标准化后每列均值：", standardized.mean(axis=0))

原始数据：
 [[ 10. 100.]
 [ 20. 120.]
 [ 30. 140.]]
均值： [ 20. 120.] shape= (2,)
标准差： [ 8.16496581 16.32993162]
标准化结果：
 [[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]
标准化后每列均值： [0. 0.]


## 9. 补充：`keepdims=True`

聚合运算默认会删除被压缩的轴。`keepdims=True` 保留长度为 1 的轴，使形状语义更明确，也常让广播更容易。

In [15]:
m = np.arange(1, 13).reshape(3, 4)
print("原始矩阵：\n", m)
row_mean = m.mean(axis=1)
row_mean_kept = m.mean(axis=1, keepdims=True)
print("普通结果：", row_mean, "shape=", row_mean.shape)
print("保留维度：\n", row_mean_kept, "shape=", row_mean_kept.shape)
print("每行减去自己的均值：\n", m - row_mean_kept)

原始矩阵：
 [[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
普通结果： [ 2.5  6.5 10.5] shape= (3,)
保留维度：
 [[ 2.5]
 [ 6.5]
 [10.5]] shape= (3, 1)
每行减去自己的均值：
 [[-1.5 -0.5  0.5  1.5]
 [-1.5 -0.5  0.5  1.5]
 [-1.5 -0.5  0.5  1.5]]


## 本课小结

- 广播让兼容形状的数组进行逐元素运算。
- 从右向左比较形状；对应维度必须相等，或其中一个是 1。
- `(n,)` 默认对齐最后一轴；`[:, None]` 可创建列向量。
- 广播通常不需要真正复制数据，比手动 `tile()` 更省内存。
- `np.broadcast_shapes()` 可预先检查结果形状。
- 聚合后需要保留轴时，可使用 `keepdims=True`。

## 本节练习

对下面的商品价格表完成：

1. 每列代表一种商品，为三种商品分别加上 `[2, 5, 10]` 的运费。
2. 每行代表一个门店，为三个门店分别乘以 `[1.0, 0.9, 0.8]` 的折扣。注意调整形状。
3. 计算每个门店的平均价格，并用 `keepdims=True` 保持二维。
4. 从每一行减去该行平均值。
5. 写出每一步参与广播的形状。

In [29]:
prices = np.array([
    [20.0, 35.0, 50.0],
    [22.0, 36.0, 48.0],
    [19.0, 34.0, 52.0],
])
# 请在这里完成练习
print("原始价格矩阵：\n", prices)
p1 = prices + np.array([2, 5, 10]) 
print("加上补贴后的价格：\n", prices + np.array([2, 5, 10]))
p2 = p1* np.array([1.0,0.9,0.8])[:, None]
print("折扣后的价格：\n",p1* np.array([1.0,0.9,0.8])[:, None])
p3 = p2.mean(axis=1, keepdims=True)
print("每行的平均价格：\n", p3)
print(p1.shape)
print(p2.shape)
print(p3.shape)
print(p2 - p3 )

原始价格矩阵：
 [[20. 35. 50.]
 [22. 36. 48.]
 [19. 34. 52.]]
加上补贴后的价格：
 [[22. 40. 60.]
 [24. 41. 58.]
 [21. 39. 62.]]
折扣后的价格：
 [[22.  40.  60. ]
 [21.6 36.9 52.2]
 [16.8 31.2 49.6]]
每行的平均价格：
 [[40.66666667]
 [36.9       ]
 [32.53333333]]
(3, 3)
(3, 3)
(3, 1)
[[-18.66666667  -0.66666667  19.33333333]
 [-15.3          0.          15.3       ]
 [-15.73333333  -1.33333333  17.06666667]]


## 参考答案

In [17]:
shipping = np.array([2.0, 5.0, 10.0])          # (3,)
discounts = np.array([1.0, 0.9, 0.8])[:, None] # (3, 1)
with_shipping = prices + shipping
discounted = with_shipping * discounts
row_means = discounted.mean(axis=1, keepdims=True)
centered = discounted - row_means
print("prices + shipping：", prices.shape, "+", shipping.shape, "→", with_shipping.shape)
print("乘门店折扣：", with_shipping.shape, "*", discounts.shape, "→", discounted.shape)
print("最终价格：\n", discounted)
print("每店平均价：\n", row_means, "shape=", row_means.shape)
print("中心化后：\n", centered)
print("每行之和接近 0：", np.allclose(centered.sum(axis=1), 0))

prices + shipping： (3, 3) + (3,) → (3, 3)
乘门店折扣： (3, 3) * (3, 1) → (3, 3)
最终价格：
 [[22.  40.  60. ]
 [21.6 36.9 52.2]
 [16.8 31.2 49.6]]
每店平均价：
 [[40.66666667]
 [36.9       ]
 [32.53333333]] shape= (3, 1)
中心化后：
 [[-18.66666667  -0.66666667  19.33333333]
 [-15.3          0.          15.3       ]
 [-15.73333333  -1.33333333  17.06666667]]
每行之和接近 0： True


---

下一课建议学习：[NumPy 迭代数组](https://www.runoob.com/numpy/numpy-iterating-over-array.html)。